# L45 · 声谱图（spectrogram）生成 — magnitude_spectrogram，imshow 热力图，读懂时频图

**目标**：把 STFT 复数矩阵转成幅度谱（magnitude spectrum）、功率谱（power spectrum）、dB 谱，再用 `imshow` 画出正确标注时频轴的热力图。

🔗 Aurora 连接：[`aurora.audio.stft.magnitude_spectrogram()`](../../src/aurora/audio/stft.py) · [`aurora.audio.mel.power_to_db()`](../../src/aurora/audio/mel.py)

← **上一课**　[L44 · 亲手写 STFT](L44_stft_implement.ipynb)

> 上节课学习了 **亲手写 STFT**：分帧 + 加窗 + FFT，与 aurora.audio.stft 对齐验证。  
> 本课将探讨 **声谱图生成**。

## 本课剧情：为什么声谱图看起来像"音乐的 X 光"？

医生用 X 光看骨骼——时间是横轴，身体部位是纵轴，亮度是密度。声谱图做的是一样的事：横轴是时间，纵轴是频率，亮度是能量。

STFT 输出的是复数矩阵 `S`（形状 `(n_frames, n_fft//2+1)`），每个元素携带幅度和相位。人耳感知的不是相位，而是**能量**——所以我们丢掉相位，只保留幅度：

```
mag = |S|           # 幅度谱，单位：任意
pow = |S|²          # 功率谱，单位：watts
dB  = 10·log₁₀(pow + ε)  # dB 谱，压缩动态范围
```

为什么要取 dB？声音的动态范围高达 120 dB（比值 10¹²）。用线性尺度画图，轻声几乎不可见；用 dB 尺度，弱音和强音都清晰可辨——这正是人耳的对数感知方式。

**读声谱图的三个坐标**：

| 轴 | 含义 | 计算方式 |
|---|---|---|
| 横轴（时间） | 第 m 帧对应 `t = m·hop/sr` 秒 | `t = np.arange(n_frames) * hop / sr` |
| 纵轴（频率） | 第 k bin 对应 `f = k·sr/n_fft` Hz | `f = np.arange(n_fft//2+1) * sr / n_fft` |
| 颜色（亮度） | dB 功率，通常 -80~0 dB 归一化 | `dB = 10·log₁₀(|S|² + 1e-9)` |

本节任务：实现 `plot_spectrogram(x, sr, win_len, hop)` — 把信号画成 dB 幅度频谱图。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine, read_wav
from aurora.audio.stft import stft, magnitude_spectrogram
from aurora.audio.mel import power_to_db

## 概念 1：幅度谱、功率谱、dB 谱

STFT 矩阵 `S` 形状为 `(n_frames, n_fft//2+1)`，每个元素是复数。

- **幅度谱**：`A = np.abs(S)`，单位是线性振幅
- **功率谱**：`P = A**2`，与能量正比
- **dB 谱**：`dB = 10 * log10(P + eps)`，eps 防 log(0)

Aurora 实现使用 eps = `1e-10`（对应 -100 dB 噪声底），并支持 `top_db` 截断防止动态范围过大。
注意：音频领域常见的另一种写法 `20 * log10(A + eps)` 与 `10 * log10(A**2 + eps)` 等价（仅在 eps 处有微小差异）。

In [ ]:
# 演示：单帧 FFT 的三种表示
SR = 16000
x = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

S = stft(x, n_fft=1024, hop_length=256)  # (n_frames, 513)
print(f"STFT shape: {S.shape}, dtype: {S.dtype}")

A = np.abs(S)                     # 幅度谱
P = A ** 2                        # 功率谱
dB = 10 * np.log10(P + 1e-10)    # dB 谱

print(f"幅度范围: [{A.min():.4f}, {A.max():.4f}]")
print(f"功率范围: [{P.min():.4e}, {P.max():.4e}]")
print(f"dB 范围:  [{dB.min():.1f}, {dB.max():.1f}] dB")

## 概念 2：`imshow` 的关键参数

`plt.imshow` 默认把矩阵第 0 行画在顶部（图像坐标系），但频率图习惯低频在下，所以必须设：

```
origin="lower"   # 第 0 行（DC / 0 Hz）在底部
aspect="auto"    # 不强制正方形像素，适配任意时频比例
extent=[t_min, t_max, f_min, f_max]  # 让坐标轴显示真实单位
```

`extent` 的顺序是 `[x_left, x_right, y_bottom, y_top]`，对应
`[0, duration_s, 0, sr/2]`。
`sr/2` 是奈奎斯特频率，即 STFT 能表示的最高频率。

In [ ]:
# 演示：origin 和 aspect 对图形的影响
SR = 16000
dur = 0.5
x = sine(440, duration=dur, sample_rate=SR)
S = stft(x, n_fft=1024, hop_length=256)
A = np.abs(S).T   # 转置：(频率箱, 帧数) → imshow 行=频率, 列=时间

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].imshow(A, aspect="auto")
axes[0].set_title("origin=upper (默认) — 频率倒置")

axes[1].imshow(A, origin="lower", aspect="auto",
               extent=[0, dur, 0, SR/2])
axes[1].set_title("origin=lower + extent — 正确")
axes[1].set_xlabel("时间 (s)")
axes[1].set_ylabel("频率 (Hz)")

plt.tight_layout()
plt.show()

## 概念 3：时间轴与频率轴的精确计算

**频率轴**：rfft 输出 `n_fft//2 + 1` 个箱，箱 `k` 对应频率：

```
f[k] = k * sr / n_fft      k = 0, 1, ..., n_fft//2
```

即 `f = np.arange(n_fft//2 + 1) * sr / n_fft`。

**时间轴**：第 `t` 帧的中心样本位于 `t * hop_length`（启用 `center=True` 时已经对齐），
换算成秒：

```
t[i] = i * hop_length / sr     i = 0, 1, ..., n_frames-1
```

`extent` 传入的只是边界，用 `[0, n_frames*hop/sr, 0, (n_fft//2)*sr/n_fft]` 足够精确。

In [ ]:
# 演示：手动计算轴刻度
SR = 16000
N_FFT = 1024
HOP = 256

f_bins = np.arange(N_FFT // 2 + 1) * SR / N_FFT   # 513 个频率
print(f"频率箱数: {len(f_bins)}")
print(f"频率分辨率: {SR/N_FFT:.2f} Hz/箱")
print(f"最高频率: {f_bins[-1]:.0f} Hz  (= SR/2 = {SR//2} Hz)")

x = sine(440, duration=1.0, sample_rate=SR)
S = stft(x, n_fft=N_FFT, hop_length=HOP)
n_frames = S.shape[0]
t_bins = np.arange(n_frames) * HOP / SR
print(f"\n帧数: {n_frames}")
print(f"时间分辨率: {HOP/SR*1000:.1f} ms/帧")
print(f"总时长覆盖: {t_bins[-1]:.3f} s")

## 1. ✏️ 实现 `plot_spectrogram(x, sr, win_len=1024, hop=256)`

**五步流程**：

| 步骤 | 代码 | 说明 |
|---|---|---|
| 1 | `S = my_stft(x, win_len, hop)` 或 `stft(x,...)` | 获取复数 STFT 矩阵 |
| 2 | `power = np.abs(S) ** 2` | 功率谱 |
| 3 | `db = 10 * np.log10(power + 1e-9)` | dB 谱（加 ε 防 log(0)） |
| 4 | `plt.imshow(db.T, origin='lower', aspect='auto', ...)` | `.T`：让频率在纵轴；`origin='lower'`：低频在下 |
| 5 | 计算时间/频率轴，设置 xlabel/ylabel | `t = np.arange(n_frames)*hop/sr`，`f = k*sr/win_len` |

**验收标准**：
- 440 Hz + 880 Hz 双音信号 → 两条清晰水平线
- `db.shape == (n_frames, win_len//2+1)`（转置前）
- 函数返回 `db` 数组（供数值验证）

In [ ]:
def plot_spectrogram(x, sr, win_len=1024, hop=256):
    """把信号画成 dB 幅度频谱图。返回 dB 矩阵以便数值验证。

    Parameters
    ----------
    x       : 1-D float array，时域信号
    sr      : 采样率（Hz）
    win_len : FFT 窗长（样本数）
    hop     : 帧移（样本数）

    Returns
    -------
    dB : ndarray, shape (n_fft//2+1, n_frames)
    """
    raise NotImplementedError("请完成 TODO 步骤 1–3，然后删除此行")  # ← 删除此行再填写实现

    # ✏️ TODO 步骤 1：调用 stft() 得到复数矩阵 S，形状 (n_frames, win_len//2+1)
    S = ...

    # ✏️ TODO 步骤 2：np.abs(S).T 得到 (频率, 时间) 幅度矩阵，再转 dB
    # 注意：此处使用 20*log10(A + 1e-8)（振幅形式），与 Aurora power_to_db 的
    # 10*log10(P + 1e-10)（功率形式）近似等价，但 eps 楼层略有差异（见 Cell 3）
    A = ...
    dB = ...

    dur = len(x) / sr

    # ✏️ TODO 步骤 3：imshow + colorbar + 轴标签（origin="lower", aspect="auto"）
    fig, ax = plt.subplots(figsize=(10, 4))
    ...

    return dB  # 返回 dB 矩阵供外部验证


In [ ]:
# 目视验证：两条水平线是否清晰
SR = 16000
x_test = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

try:
    student_dB = plot_spectrogram(x_test, SR)
    if student_dB is not None and hasattr(student_dB, "shape"):
        assert student_dB.shape[0] == 513, (
            f"频率箱数应为 513，实际 {student_dB.shape[0]}"
        )
        assert student_dB.max() > -20, (
            f"最大 dB 值 {student_dB.max():.1f} dB 过低，检查幅度计算"
        )
        print(f"✅ plot_spectrogram 验证通过：shape={student_dB.shape}，"
              f"dB.max()={student_dB.max():.1f} dB")
    else:
        print("⚠️  plot_spectrogram 返回 None，请确保函数末尾有 return dB")
except NotImplementedError:
    print("⬜ 未实现：删除 raise NotImplementedError 行并填写 TODO")

# 数值检查（参考实现，用于 Cell 13 的色阶范围）
from aurora.audio.stft import stft as _stft
S_ref = _stft(x_test, n_fft=1024, hop_length=256)
A_ref = np.abs(S_ref).T
dB_ref = 20 * np.log10(A_ref + 1e-8)

assert dB_ref.max() > -20, "亮线 dB 值过低，检查幅度计算"
assert dB_ref.shape[0] == 513, f"频率箱数应为 513，实际 {dB_ref.shape[0]}"
print(f"✅ 参考实现：shape={dB_ref.shape}，dB_ref.max()={dB_ref.max():.1f} dB")
print("   （440 Hz 和 880 Hz 处应有清晰水平亮线）")


## 参数实验：改变 win_len 和 hop 观察时频分辨率权衡

`win_len`（窗长）和 `hop`（帧移）控制**时频分辨率的权衡**：

| 参数 | 变大时 | 代价 |
|------|--------|------|
| `win_len` 增大 | 频率分辨率（frequency resolution）提高（箱更窄） | 时间分辨率（time resolution）下降（帧覆盖更长） |
| `hop` 减小 | 时间分辨率提高（帧更密） | 计算量增加 |

**预期现象**：
- `win_len=256, hop=64`：水平线变粗（频率模糊），时间轴很密
- `win_len=2048, hop=512`：水平线很细（频率清晰），瞬态事件变模糊

In [ ]:
SR = 16000
x_exp = sine(440, duration=1.0, sample_rate=SR) + sine(880, duration=1.0, sample_rate=SR)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (256,  64,  "win=256 hop=64\n（时间清晰，频率模糊）"),
    (1024, 256, "win=1024 hop=256\n（均衡，默认）"),
    (2048, 512, "win=2048 hop=512\n（频率清晰，时间模糊）"),
]

for ax, (win, hop, title) in zip(axes, configs):
    S = stft(x_exp, n_fft=win, hop_length=hop)
    A = np.abs(S).T
    dB = 20 * np.log10(A + 1e-8)
    dur = len(x_exp) / SR
    im = ax.imshow(dB, origin="lower", aspect="auto",
                   extent=[0, dur, 0, SR/2], cmap="inferno",
                   vmin=dB_ref.max()-60, vmax=dB_ref.max())
    ax.set_title(title)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("频率 (Hz)")
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print("✅ 观察三列图中亮线宽度的变化")

## 本课收束

本课实现了 `plot_spectrogram()`，输出以 dB 为单位的时频热力图。
核心操作链为 `stft → np.abs → 平方/对数 → imshow`，其中 `origin="lower"` 和 `extent` 是正确对齐时频轴的关键。
这些操作已封装进 Aurora 的 `aurora.audio.stft.magnitude_spectrogram()` 和 `aurora.audio.mel.power_to_db()`，下节课（L46）将在此基础上构建 Mel 滤波器组，把线性频率轴压缩成听觉感知的 Mel 刻度。

## Aurora 连接验证 — magnitude_spectrogram 与 power_to_db

Aurora 封装的 `magnitude_spectrogram()` 和 `power_to_db()` 与本课手写链等价，但使用功率形式：

| 形式 | 公式 | eps | dB 底 |
|------|------|-----|-------|
| 振幅形式（本课练习） | `20·log10(A + 1e-8)` | 1e-8 | ≈ −160 dB |
| 功率形式（Aurora API） | `10·log10(P + 1e-10)` | 1e-10 | ≈ −100 dB |

两者对 A≥0.01 的信号数值近似相等；仅在接近静音（A≈0）时 eps 楼层不同。
下方代码验证两者在 440+880 Hz 测试信号上的最大差异 < 1 dB。

In [ ]:
# Aurora 连接：验证 magnitude_spectrogram / power_to_db 与手写链的一致性
SR = 16000
x_verify = sine(440, duration=0.5, sample_rate=SR) + sine(880, duration=0.5, sample_rate=SR)

# 手写链（振幅形式，本课练习公式）
S_hand = stft(x_verify, n_fft=1024, hop_length=256)
A_hand = np.abs(S_hand).T
dB_hand = 20 * np.log10(A_hand + 1e-8)

# Aurora API（功率形式）
dB_aurora = power_to_db(np.abs(S_hand).T ** 2)

# 对信号能量较高区域（dB > -40）比较
mask = dB_hand > -40
diff = np.abs(dB_hand[mask] - dB_aurora[mask])
print(f"手写链 dB 范围: [{dB_hand.min():.1f}, {dB_hand.max():.1f}] dB")
print(f"Aurora dB 范围: [{dB_aurora.min():.1f}, {dB_aurora.max():.1f}] dB")
print(f"信号区域最大差异: {diff.max():.2f} dB（< 1 dB 即视为等价）")
assert diff.max() < 1.0, f"差异过大: {diff.max():.2f} dB"
print("✅ Aurora API 与手写链在信号区域等价（eps 仅影响静音底噪）")

# 同时演示 magnitude_spectrogram 直接调用
dB_direct = magnitude_spectrogram(x_verify)
print(f"magnitude_spectrogram() 输出 shape: {dB_direct.shape}")
print("✅ magnitude_spectrogram 可直接替代本课练习的完整实现链")


## ✏️ 白板挑战：声谱图手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：`sr=8000, win_len=256, hop=128, duration=1.0s`：
- 总采样点 N = sr × duration = ?
- n_frames = 1 + (N - win_len) // hop = ?
- 频谱图输出 shape（转置前）= ?

**问 2**：`S[5, 10]` 对应的时间（秒）和频率（Hz）各是多少？  
（`sr=8000, win_len=256, hop=128`）

**问 3**：为什么 `plt.imshow` 需要 `.T`（转置）和 `origin='lower'`？  
（提示：STFT 矩阵行=时间，列=频率；imshow 默认纵轴原点在哪？）

**问 4**：`dB = 10·log₁₀(|S|² + ε)` 中的 ε（如 1e-9）起什么作用？  
当 `|S|=0` 时不加 ε 会发生什么？

**问 5**：440 Hz 在 `win_len=256, sr=8000` 的频谱图中，出现在第几个 bin？  
（`k = round(f / Δf) = round(440 * 256 / 8000) = ?`）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：shape 计算
sr1, win1, hop1, dur1 = 8000, 256, 128, 1.0
N1 = int(sr1 * dur1)
n_frames1 = 1 + (N1 - win1) // hop1
n_bins1 = win1 // 2 + 1
assert N1 == 8000
assert n_frames1 == 62
assert n_bins1 == 129
print(f"Q1 ✅  N={N1}, n_frames={n_frames1}, 频谱shape(转置前)=({n_frames1},{n_bins1})")

# 问2：帧/频率对应时间和Hz
m, k = 5, 10
t_sec = m * hop1 / sr1
f_hz  = k * sr1 / win1
assert abs(t_sec - 0.08) < 1e-10
assert f_hz == 312.5
print(f"Q2 ✅  S[5,10] → t={t_sec:.3f}s，f={f_hz:.1f} Hz")

# 问3：imshow 转置和 origin
# 验证：imshow 默认 origin='upper'（行0在顶部）；频谱图需要低频在下
test_mat = np.arange(6).reshape(2, 3)  # 行=时间(2帧)，列=频率(3bin)
# 转置后：行=频率(3)，列=时间(2)；origin='lower'让row0(低频)在下
assert test_mat.T.shape == (3, 2)  # 转置后行=频率
print(f"Q3 ✅  .T 让行=频率（imshow按行画）；origin='lower'让row0低频在下（默认upper→低频在上）")

# 问4：ε 防 log(0)
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    log_zero = np.log10(0.0)
assert np.isinf(log_zero)  # log(0) = -inf
log_eps = 10 * np.log10(0.0 + 1e-9)
assert np.isfinite(log_eps) and log_eps < 0
print(f"Q4 ✅  log₁₀(0)={log_zero}（−inf），加ε→{log_eps:.1f} dB（有限值，代表极弱信号）")

# 问5：440Hz 对应的 bin
f_target, win5, sr5 = 440, 256, 8000
k_bin = round(f_target * win5 / sr5)
f_actual = k_bin * sr5 / win5
assert k_bin == 14
assert f_actual == 437.5
print(f"Q5 ✅  440 Hz → k={k_bin}号bin（精确频率={f_actual} Hz，Δf={sr5/win5:.1f} Hz/bin）")
print("\n🎉 声谱图白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l45_review = {
    "db_formula":              None,  # 记住 dB=10·log₁₀(|S|²+ε)？True/False
    "plot_spectrogram_impl":   None,  # plot_spectrogram 实现，两条线清晰可见？True/False
    "transpose_and_origin":    None,  # 理解为何要 .T 和 origin='lower'？True/False
    "time_freq_axis_calc":     None,  # 能手算 t=m·hop/sr 和 f=k·sr/win？True/False
    "whiteboard_passed":       None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l45_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l45_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L45 全部通关！进入 L46：Mel 频率尺度')

---

→ **下一课**　[L46 · Mel 频率尺度](L46_mel.ipynb)

> 下节课将学习 **Mel 频率尺度**：人耳对数感知，mel = 2595·log₁₀(1+f/700)，三角滤波器。